In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F

In [ ]:
# --- 1. Hyperparameters ---
batch_size = 32      # How many independent sequences will we process in parallel?
block_size = 64      # What is the maximum context length for predictions?
max_iters = 3000     # Total training steps
learning_rate = 1e-3
device = 'cuda' if torch.cuda.is_available() else 'cpu' # Auto-detect T4 GPU
n_embd = 128         # Number of embedding dimensions
n_head = 4           # Number of attention heads
n_layer = 4          # Number of transformer blocks
dropout = 0.1

print(f"Using device: {device}")

Using device: cuda


In [3]:
# --- 2. Data Loading & Cleaning ---
# For this example, we'll use a tiny snippet of Shakespeare
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

# Simple Tokenizer
chars = sorted(list(set(text)))
vocab_size = len(chars)
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

# Train/Val splits
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

def get_batch(split):
    data_set = train_data if split == 'train' else val_data
    ix = torch.randint(len(data_set) - block_size, (batch_size,))
    x = torch.stack([data_set[i:i+block_size] for i in ix])
    y = torch.stack([data_set[i+1:i+block_size+1] for i in ix])
    return x.to(device), y.to(device)

--2026-03-06 10:22:24--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.108.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.008s  

2026-03-06 10:22:25 (134 MB/s) - ‘input.txt’ saved [1115394/1115394]



In [4]:
# --- 3. The Architecture ---
class TransformerBlock(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = nn.MultiheadAttention(n_embd, n_head, dropout=dropout, batch_first=True)
        self.ffwd = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        # Attention + Residual Connection
        attn_output, _ = self.sa(self.ln1(x), self.ln1(x), self.ln1(x), need_weights=False)
        x = x + attn_output
        # Feed-Forward + Residual Connection
        x = x + self.ffwd(self.ln2(x))
        return x

class ProtoLLM(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[TransformerBlock(n_embd, n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device))
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:] # Crop context
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

In [5]:
# --- 4. Training Loop ---
model = ProtoLLM().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):
    xb, yb = get_batch('train')
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    if iter % 500 == 0:
        print(f"step {iter}: loss {loss.item():.4f}")

step 0: loss 4.4264
step 500: loss 0.0455
step 1000: loss 0.0427
step 1500: loss 0.0504
step 2000: loss 0.0332
step 2500: loss 0.0362


In [6]:
# --- 5. Final Generation ---
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print("\n--- GENERATED TEXT ---")
print(decode(model.generate(context, max_new_tokens=200)[0].tolist()))


--- GENERATED TEXT ---



































































EL

NJM








O




O




N

MEN
LOLOLLOU
LOMENES

WONO:
A OLLULLELLLLAULLOLLLLELLLLLLLLLLLLLLLLLLLLLLLLLLLULLLLLLLLLLLELLLYLLLLLLLL
